In [145]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

from tensorgrad import Tensor, Layer, MLP, unbroadcast, gradcheck

In [146]:
BATCH_SIZE = 100

train_data = datasets.MNIST(root='../data',
                            download=True,
                            train=True,
                            transform=transforms.Lambda(lambda x : np.array(x).flatten()))

test_data = datasets.MNIST(root='../data',
                           download=True,
                           train=False,
                           transform=transforms.Lambda(lambda x : np.array(x).flatten()))

x_train, y_train = [], []
x_batch, y_batch = [], []
x_val, y_val = [], []

for i, (x, y) in enumerate(train_data, 1):
    if i > 0 and i % BATCH_SIZE == 0:
        if i <= 50000:
            x_train.append(Tensor(np.array(x_batch)))
            y_train.append(Tensor(np.array(y_batch)))
        else:
            x_val.append(Tensor(np.array(x_batch)))
            y_val.append(Tensor(np.array(y_batch)))
        x_batch, y_batch = [], []

    x_batch.append(x / 255)
    y_batch.append(np.eye(10)[y])

x_test, y_test = [], []
x_batch, y_batch = [], []

for i, (x, y) in enumerate(test_data, 1):
    if i > 0 and i % BATCH_SIZE == 0:
        x_test.append(Tensor(np.array(x_batch)))
        y_test.append(Tensor(np.array(y_batch)))
        x_batch, y_batch = [], []
    x_batch.append(x / 255)
    y_batch.append(np.eye(10)[y])

len(x_train), len(y_train), len(x_val), len(y_val), len(x_test), len(y_test)

(500, 500, 100, 100, 100, 100)

In [147]:
EPOCHS = 100
LEARNING_RATE = 1e-2

model = MLP()

for epoch in range(1, EPOCHS + 1):
    for x, y in zip(x_train, y_train):
        y_pred = model(x)
        loss = y_pred.cross_entropy(y.data)
        model.zero_grad()
        loss.backward()
        model.step(LEARNING_RATE)

    if epoch % 10 == 0:
        print(f'epoch: {epoch}, loss: {loss.data:.4f}')

epoch: 10, loss: 0.2479
epoch: 20, loss: 0.1501
epoch: 30, loss: 0.0968
epoch: 40, loss: 0.0671
epoch: 50, loss: 0.0490
epoch: 60, loss: 0.0381
epoch: 70, loss: 0.0310
epoch: 80, loss: 0.0257
epoch: 90, loss: 0.0213
epoch: 100, loss: 0.0178


In [148]:
total = len(test_data)
correct = 0
for x, y in zip(x_test, y_test):
    y_pred = model(x)
    correct += np.sum(np.argmax(y.data, -1) == np.argmax(y_pred.data, -1))

print(f'loss: {loss.data:.4f}, accuracy: {correct}/{total} = {correct/total*100:.4f}%')

loss: 0.0178, accuracy: 9781/10000 = 97.8100%


In [149]:
class Torch_MLP(nn.Module):

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 64)
        self.out = nn.Linear(64, 10)

    def __call__(self, x):
        x = self.fc1(x).relu()
        x = self.fc2(x).relu()
        x = self.out(x)

        return x

criterion = nn.CrossEntropyLoss()

torch_model = Torch_MLP()

torch_model.fc1.weight = torch.nn.Parameter(torch.tensor(model.fc1.weights.data).T)
torch_model.fc2.weight = torch.nn.Parameter(torch.tensor(model.fc2.weights.data).T)
torch_model.out.weight = torch.nn.Parameter(torch.tensor(model.out.weights.data).T)

torch_model.fc1.bias = torch.nn.Parameter(torch.tensor(model.fc1.bias.data).T)
torch_model.fc2.bias = torch.nn.Parameter(torch.tensor(model.fc2.bias.data).T)
torch_model.out.bias = torch.nn.Parameter(torch.tensor(model.out.bias.data).T)

for x, y in zip(x_test, y_test):
    y_pred = model(x)
    loss = y_pred.cross_entropy(y.data)

print(loss)

torch_loss = 0

for x, y in zip(x_test, y_test):
    x = torch.tensor(x.data)
    y = torch.tensor(y.data)
    y_pred = torch_model(x)
    torch_loss = criterion(y_pred, y)

print(torch_loss.detach().numpy())

0.059494042733933473
0.059494042733933376


In [165]:
model.zero_grad()
y_pred = model(x_test[0])
loss = y_pred.cross_entropy(y_test[0].data)
loss.backward()
np.sum(model.fc1.weights.grad)

np.float64(4.184771815322529)

In [166]:
torch_model.zero_grad()
x = torch.tensor(x_test[0].data)
y = torch.tensor(y_test[0].data)
y_pred = torch_model(x)
torch_loss = criterion(y_pred, y)
torch_loss.backward()
np.allclose(model.fc1.weights.grad, torch_model.fc1.weight.grad.T.detach().numpy())

True